In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

# ------------------------------------------------------------------
# Paths: replace these with the actual JRA and SPEEDY files/patterns.
# open_mfdataset accepts wildcards.
# ------------------------------------------------------------------
JRA_PATH = Path("/leonardo_scratch/fast/ICT26_ESP/ntilinin/INPUT/OMIP/tas_input4MIPs_atmosphericState_OMIP_MRI-JRA55-do-1-5-0_gr_199001010000-199012312100.nc")
SPEEDY_PATH = Path("/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/tas_SPEEDY_1990.nc")

JRA_VARIABLE = "tas"
SPEEDY_VARIABLE = "tas"

# Scientific comparison target:
# "jra"    -> interpolate SPEEDY to the JRA grid
# "speedy" -> interpolate JRA to the SPEEDY grid
REGRID_TO = "jra"

# Use only for a quick test before running over all files.
MAX_TIME_RECORDS = None

In [2]:
def open_any(path):
    path = str(path)
    if any(char in path for char in "*?[]"):
        return xr.open_mfdataset(
            path,
            combine="by_coords",
            parallel=False,
            chunks={"time": 120},
            decode_times=True,
            use_cftime=True,
        )

    return xr.open_dataset(
        path,
        chunks={"time": 120},
        decode_times=True,
        use_cftime=True,
    )

jra = open_any(JRA_PATH)
speedy = open_any(SPEEDY_PATH)

if MAX_TIME_RECORDS is not None:
    jra = jra.isel(time=slice(0, MAX_TIME_RECORDS))
    speedy = speedy.isel(time=slice(0, MAX_TIME_RECORDS))

print("JRA")
print(jra)
print("\nSPEEDY")
print(speedy)


/scratch_local/ipykernel_408533/1764846578.py:13: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(


JRA
<xarray.Dataset> Size: 2GB
Dimensions:    (time: 2920, bnds: 2, lat: 320, lon: 640)
Coordinates:
  * time       (time) object 23kB 1990-01-01 00:00:00 ... 1990-12-31 21:00:00
  * lat        (lat) float64 3kB -89.57 -89.01 -88.45 ... 88.45 89.01 89.57
  * lon        (lon) float64 5kB 0.0 0.5625 1.125 1.688 ... 358.3 358.9 359.4
    height     float64 8B ...
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) object 47kB dask.array<chunksize=(120, 2), meta=np.ndarray>
    lat_bnds   (lat, bnds) float64 5kB dask.array<chunksize=(320, 2), meta=np.ndarray>
    lon_bnds   (lon, bnds) float64 10kB dask.array<chunksize=(640, 2), meta=np.ndarray>
    tas        (time, lat, lon) float32 2GB dask.array<chunksize=(120, 320, 640), meta=np.ndarray>
Attributes: (12/36)
    Conventions:         CF-1.7 CMIP-6.2
    activity_id:         input4MIPs
    cell_measures:       area: areacella
    comment:             Based on JRA-55 reanalysis (1958-01 to 2020-07)
    contact

/scratch_local/ipykernel_408533/1764846578.py:13: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(


In [3]:
# Standardize coordinate names
COORD_ALIASES = {
    "longitude": "lon",
    "latitude": "lat",
    "xt_ocean": "lon",
    "yt_ocean": "lat",
    "nav_lon": "lon",
    "nav_lat": "lat",
}

def standardize_coords(ds):
    rename = {
        old: new
        for old, new in COORD_ALIASES.items()
        if old in ds.coords or old in ds.dims
    }
    return ds.rename(rename)

jra = standardize_coords(jra)
speedy = standardize_coords(speedy)

for name, ds in {"JRA": jra, "SPEEDY": speedy}.items():
    missing = {"time", "lat", "lon"} - set(ds.coords)
    if missing:
        raise KeyError(f"{name} is missing coordinates: {missing}")

In [4]:
# Structural comparison
def calendar_name(time):
    return time.encoding.get(
        "calendar",
        time.attrs.get("calendar", "unknown"),
    )

def timestep_hours(time):
    if time.size < 2:
        return np.nan

    differences = np.diff(time.values)
    hours = np.array([
        d.total_seconds() / 3600.0
        if hasattr(d, "total_seconds")
        else d / np.timedelta64(1, "h")
        for d in differences
    ])
    return np.unique(hours)

def dataset_summary(ds, variable):
    da = ds[variable]
    return {
        "variable": variable,
        "dimensions": str(da.dims),
        "shape": str(da.shape),
        "dtype": str(da.dtype),
        "units": da.attrs.get("units"),
        "standard_name": da.attrs.get("standard_name"),
        "calendar": calendar_name(ds.time),
        "time_start": str(ds.time.values[0]),
        "time_end": str(ds.time.values[-1]),
        "time_records": ds.sizes["time"],
        "time_step_hours": str(timestep_hours(ds.time)),
        "lat_size": ds.sizes["lat"],
        "lat_first": float(ds.lat.values[0]),
        "lat_last": float(ds.lat.values[-1]),
        "lon_size": ds.sizes["lon"],
        "lon_first": float(ds.lon.values[0]),
        "lon_last": float(ds.lon.values[-1]),
        "fill_value": da.encoding.get("_FillValue"),
        "chunks": str(da.chunks),
    }

structural = pd.DataFrame(
    {
        "JRA": dataset_summary(jra, JRA_VARIABLE),
        "SPEEDY": dataset_summary(speedy, SPEEDY_VARIABLE),
    }
)

#structural

In [5]:
# Attribute comparison

def compare_dicts(left, right):
    keys = sorted(set(left) | set(right))
    rows = []

    for key in keys:
        lv = left.get(key, "<missing>")
        rv = right.get(key, "<missing>")
        rows.append({
            "attribute": key,
            "JRA": repr(lv),
            "SPEEDY": repr(rv),
            "equal": lv == rv,
        })

    return pd.DataFrame(rows)

variable_attributes = compare_dicts(
    jra[JRA_VARIABLE].attrs,
    speedy[SPEEDY_VARIABLE].attrs,
)

global_attributes = compare_dicts(
    jra.attrs,
    speedy.attrs,
)

#print("Variable attributes")
#display(variable_attributes)

#print("Global attributes")
#display(global_attributes)

In [6]:
# Coordinate comparison

coordinate_rows = []

for coord in ("time", "lat", "lon"):
    coordinate_rows.append({
        "coordinate": coord,
        "JRA size": jra.sizes[coord],
        "SPEEDY size": speedy.sizes[coord],
        "same size": jra.sizes[coord] == speedy.sizes[coord],
        "same values": (
            jra.sizes[coord] == speedy.sizes[coord]
            and np.array_equal(jra[coord].values, speedy[coord].values)
        ),
        "JRA attrs": dict(jra[coord].attrs),
        "SPEEDY attrs": dict(speedy[coord].attrs),
    })

coordinate_comparison = pd.DataFrame(coordinate_rows)
#coordinate_comparison